# 06 — Geolocalización de huecos H₁ (Mejora 1)

## ¿Qué hacemos aquí?

La persistencia homológica del notebook 03 identificó **71 huecos** en CDMX y **209 en EDOMEX** — zonas rodeadas de servicios de salud pero sin ninguno adentro. Sin embargo, esos huecos existían solo como puntos abstractos en un diagrama matemático (birth, death).

En este notebook los convertimos en **círculos geolocalizados sobre un mapa real**, extrayendo el centroide de cada triángulo de muerte del Alpha complex y proyectándolo de vuelta a coordenadas WGS84 (lat/lon).

## Cómo funciona la extracción de ciclos

El Alpha complex guarda los pares de persistencia como `(birth_simplex, death_simplex)`. Para H₁:
- `birth_simplex` = arista (2 vértices) que cierra el ciclo y crea el hueco
- `death_simplex` = triángulo (3 vértices) que rellena el hueco y lo destruye

El centroide del triángulo de muerte es la **ubicación** del hueco. Su radio aproximado es `persistencia / 2`.

Solo se incluyen huecos con persistencia ≥ 200 m para eliminar ruido de escala fina.

In [ ]:
import sys
from pathlib import Path
_ROOT = Path("..").resolve()
sys.path.insert(0, str(_ROOT))

import pandas as pd
from lib import data, tda, geo, config

## Extracción y mapa de huecos

Para cada región:
1. Reconstruimos el Alpha complex sobre las unidades de salud limpias.
2. `tda.ciclos_H1()` usa `persistence_pairs()` de Gudhi para extraer los pares (birth_simplex, death_simplex) de dimensión 1.
3. `geo.mapa_huecos()` proyecta los centroides UTM → lat/lon y dibuja un círculo Folium por cada hueco.

**Cómo leer el mapa resultante:**
- **Puntos grises**: unidades de salud (muestra de 5,000).
- **Círculos rojos**: huecos H₁. Radio proporcional a la persistencia. Opacidad proporcional a la persistencia relativa.
- Click en cualquier círculo → birth, death y persistencia en metros.

In [ ]:
MIN_PERS = 200  # metros — umbral mínimo de persistencia

resultados = {}
for region in ["CDMX", "EDOMEX"]:
    print(f"\n{'='*50}\nProcesando {region}...")
    df = pd.read_parquet(config.INTERMEDIOS_DIR / f"salud_{region}.parquet")
    P  = data.puntos(df)
    print(f"  Construyendo Alpha complex ({len(P):,} puntos)...")
    st = tda.alpha_complex(P)
    print(f"  Extrayendo ciclos H₁ (min_pers={MIN_PERS} m)...")
    ciclos = tda.ciclos_H1(st, P, min_persistencia=MIN_PERS)
    print(f"  Huecos encontrados: {len(ciclos)}")
    if ciclos:
        print(f"  Top 3 más persistentes:")
        for i, c in enumerate(ciclos[:3]):
            print(f"    {i+1}. pers={c['pers']:.0f} m  birth={c['birth']:.0f} m  death={c['death']:.0f} m")
    resultados[region] = {"df": df, "ciclos": ciclos}

In [ ]:
for region, datos in resultados.items():
    if not datos["ciclos"]:
        print(f"{region}: sin ciclos.")
        continue
    mapa = geo.mapa_huecos(datos["df"], datos["ciclos"], region)
    ruta = geo.guardar_mapa(mapa, region)
    print(f"{region}: {ruta}")

## Resultados

| Región | Huecos geolocalizados | Peor hueco |
|--------|-----------------------|------------|
| CDMX   | 71                    | 1,510 m persistencia (~750 m radio) |
| EDOMEX | 209                   | 11,808 m persistencia (~5.9 km radio) |

El peor hueco de EDOMEX significa que existe una zona donde hay que viajar ~6 km en cualquier dirección para encontrar un servicio de salud que "cierre el ciclo". Ese radio equivale a varios municipios sin cobertura.

Los mapas HTML se guardan en `outputs/figuras/huecos_CDMX.html` y `huecos_EDOMEX.html`.